# TP Vision (LA Maxime et HORION Antoine)

Extraction des images et des classificateurs

In [ ]:
import cv2
import operator as op
import sys
import copy
import numpy as np

face_cascade = cv2.CascadeClassifier("classifieurs/haarcascade_frontalface_alt2.xml")
profile_cascade = cv2.CascadeClassifier("classifieurs/haarcascade_profileface.xml")

img_test = cv2.imread("Images de test pour la decetion de visages/image_test.jpg")
img_test = cv2.resize(img_test,(0,0),fx=0.25,fy=0.25)
ff = cv2.imread("Images de test pour la decetion de visages/ff.jpg")
tt = cv2.imread("Images de test pour la decetion de visages/tt.png")

img_list = [img_test,ff,tt]

cv2.imshow("img_test",img_test) 
cv2.imshow("ff",ff)
cv2.imshow("tt",tt)

cv2.waitKey(0)
cv2.destroyAllWindows()

Définition des fonctions

In [2]:
def overlap(face,profile): #fonction de detection des chavauchements
    x1,y1,w1,h1 = face
    x2,y2,w2,h2 = profile

    return not (x1 + w1 < x2 or x2 + w2 < x1 or y1 + h1 < y2 or y2 + h2 < y1)


def detection_visages (face_cascade, profile_cascade, img_, compteur_visage = True):
    
    img_color = copy.deepcopy(img_)
    img = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
    img_mir = cv2.flip(img,1)
    faces = face_cascade.detectMultiScale(img,1.15,6)
    profiles = profile_cascade.detectMultiScale(img,1.1,6) 
    
    # Détection des profils pour l'image miroir
    profiles_mir = profile_cascade.detectMultiScale(img_mir,1.1,3)
    w_img = len(img[0])
    h_img = len(img)
    # Si un profil est détecté dans l'image miroir on place les coordinnées du rectangle au 
    # bon endroit sur l'image de base.
    for p in profiles_mir:
        p[0] = w_img - p[0] - p[2]
    
    profiles_all = list(profiles) + list(profiles_mir)
    
    no_ol_profiles =[] #detection des chevauchements
    for p in profiles_all:
        if not any(overlap(f,p) for f in faces):
            no_ol_profiles.append(p)
            
    

    list_faces = list(faces) + no_ol_profiles
    
    face_count = 0

    for(x,y,w,h) in list_faces: #dessin des rectangles
        x2,y2 = x+w , y+h
        cv2.rectangle(img_color, (x, y), (x2, y2), (0, 0, 255), thickness=4)
        # numérotation des visages
        text = f"Visage {face_count+1}"
        cv2.putText(img_color, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        face_count += 1
    
    if compteur_visage :
        print("nombre de visage dans l'image : ", face_count)
            
    return img_color


Détection des visages sur images fixes

In [3]:
idx = 0
for img in img_list:
    img_faces = detection_visages(face_cascade, profile_cascade, img)

    cv2.imshow(f"Image_{idx}", img_faces)
    cv2.imwrite(f"image {idx}.png",img_faces)

    idx +=1


cv2.waitKey(0)
cv2.destroyAllWindows()

nombre de visage dans l'image :  6
nombre de visage dans l'image :  5
nombre de visage dans l'image :  2


Détection des visages sur flux vidéo

In [4]:
cap = cv2.VideoCapture(0)
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi',fourcc, 24.0, (1280,720)) 

while(cap.isOpened()):
    ret, frame = cap.read()
    if ret == True:
        frame = cv2.flip(frame,1)
        
        frame = detection_visages(face_cascade, profile_cascade, frame, False)
            
        out.write(frame)
        cv2.imshow('frame', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):  # Il faut appuyer sur "q" pour fermer la fenêtre
            break
    else:
        break
cap.release()
out.release()
cv2.destroyAllWindows()